# Ray AI
Ray AI Runtime (AIR) is a scalable and unified toolkit for ML applications. AIR enables simple scaling of individual workloads, end-to-end workflows, and popular ecosystem frameworks, all in just Python.

## First, let’s start by loading a dataset from storage:

In [2]:
import ray

# can define runtime_env to define dependencies etc
runtime_env = {"working_dir": "/mnt/shared_data", "pip": ["numpy"]}

# Let's start Ray
if ray.is_initialized():
    ray.shutdown()
ray.init(runtime_env=runtime_env)

2026-02-25 11:30:45,561	INFO worker.py:1669 -- Using address ray://10.10.1.98:10001 set in the environment variable RAY_ADDRESS
2026-02-25 11:30:45,564	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: log_to_driver
2026-02-25 11:30:45,798	INFO packaging.py:691 -- Creating a file package for local module '/mnt/shared_data'.
2026-02-25 11:30:45,804	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_8ec60382a4649345.zip' (0.14MiB) to Ray cluster...
2026-02-25 11:30:45,810	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_8ec60382a4649345.zip'.
SIGTERM handler is not set because current thread is not the main thread.


Python version:,3.11.6
Ray version:,2.54.0
Dashboard:,http://10.10.1.98:8265


### Set up your dataset and model.

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

def get_dataset():
    return datasets.FashionMNIST(
        root="/mnt/shared_data/datasets",
        train=True,
        download=True,
        transform=ToTensor(),
    )

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, inputs):
        inputs = self.flatten(inputs)
        logits = self.linear_relu_stack(inputs)
        return logits

### Now define your single-worker PyTorch training function.

In [6]:
def train_func():
    num_epochs = 20
    batch_size = 64

    dataset = get_dataset()
    dataloader = DataLoader(dataset, batch_size=batch_size)

    model = NeuralNetwork()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

    for epoch in range(num_epochs):
        for inputs, labels in dataloader:
            optimizer.zero_grad()
            pred = model(inputs)
            loss = criterion(pred, labels)
            loss.backward()
            optimizer.step()
        print(f"epoch: {epoch}, loss: {loss.item()}")

### This training function can be executed with:

In [7]:
%%time
train_func()

100%|██████████| 26.4M/26.4M [00:00<00:00, 47.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 4.61MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 32.3MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 6.76MB/s]


epoch: 0, loss: 0.9102227687835693
epoch: 1, loss: 0.7611222863197327
epoch: 2, loss: 0.7059283256530762
epoch: 3, loss: 0.6834201216697693
epoch: 4, loss: 0.6643202900886536
epoch: 5, loss: 0.6419609785079956
epoch: 6, loss: 0.618054211139679
epoch: 7, loss: 0.5953715443611145
epoch: 8, loss: 0.574187159538269
epoch: 9, loss: 0.5524927973747253
epoch: 10, loss: 0.5328249931335449
epoch: 11, loss: 0.5143214464187622
epoch: 12, loss: 0.49564534425735474
epoch: 13, loss: 0.4793558716773987
epoch: 14, loss: 0.46207544207572937
epoch: 15, loss: 0.44402119517326355
epoch: 16, loss: 0.4267239272594452
epoch: 17, loss: 0.4118867814540863
epoch: 18, loss: 0.39663124084472656
epoch: 19, loss: 0.3842875361442566
CPU times: user 39min 6s, sys: 3.87 s, total: 39min 10s
Wall time: 2min 32s


Convert this to a distributed multi-worker training function.

Use the ray.train.torch.prepare_model and ray.train.torch.prepare_data_loader utility functions to set up your model and data for distributed training. This automatically wraps the model with DistributedDataParallel and places it on the right device, and adds DistributedSampler to the DataLoaders.

In [8]:
import ray.train.torch

def train_func_distributed():
    num_epochs = 20
    batch_size = 64

    dataset = get_dataset()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    dataloader = ray.train.torch.prepare_data_loader(dataloader)

    model = NeuralNetwork()
    model = ray.train.torch.prepare_model(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

    for epoch in range(num_epochs):
        if ray.train.get_context().get_world_size() > 1:
            dataloader.sampler.set_epoch(epoch)

        for inputs, labels in dataloader:
            optimizer.zero_grad()
            pred = model(inputs)
            loss = criterion(pred, labels)
            loss.backward()
            optimizer.step()
        print(f"epoch: {epoch}, loss: {loss.item()}")

In [12]:
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig  # <-- Import RunConfig

# For GPU Training, set `use_gpu` to True.
use_gpu = True

trainer = TorchTrainer(
    train_func_distributed,
    scaling_config=ScalingConfig(num_workers=4, use_gpu=use_gpu),
    run_config=RunConfig(
        storage_path="/mnt/shared_data/ray_results",
        name="fashion_mnist_experiment"
    )
)

In [13]:
%%time
results = trainer.fit()

(TrainController pid=31912) Attempting to start training worker group of size 4 with the following resources: [{'GPU': 1}] * 4
(RayTrainWorker pid=20395, ip=10.10.4.23) Setting up process group for: env:// [rank=0, world_size=4]
(RayTrainWorker pid=32020) Moving model to device: cuda:0
(RayTrainWorker pid=32020) Wrapping provided model in DistributedDataParallel.
(TrainController pid=31912) Started training worker group of size 4: 
(TrainController pid=31912) - (ip=10.10.4.23, pid=20395) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=31912) - (ip=10.10.1.34, pid=25424) world_rank=1, local_rank=0, node_rank=1
(TrainController pid=31912) - (ip=10.10.3.63, pid=20398) world_rank=2, local_rank=0, node_rank=2
(TrainController pid=31912) - (ip=10.10.1.98, pid=32020) world_rank=3, local_rank=0, node_rank=3
(RayTrainWorker pid=20398, ip=10.10.3.63) Moving model to device: cuda:0
(RayTrainWorker pid=20398, ip=10.10.3.63) Wrapping provided model in DistributedDataParallel.
(RayTrain

CPU times: user 1.05 s, sys: 454 ms, total: 1.51 s
Wall time: 1min 27s


In [14]:
results

Result(metrics=None, checkpoint=None, error=None, path='/mnt/shared_data/ray_results/fashion_mnist_experiment', metrics_dataframe=None, best_checkpoints=[], _storage_filesystem=<pyarrow._fs.LocalFileSystem object at 0x7de1941313b0>)

In [15]:
# shutdown in the end
ray.shutdown()